# 98 — Build virtual T1 integration plan (SAFE, v2)

This notebook converts notebook 97's source clusters and pairwise timing results
into an explicit, reproducible plan for one virtual T1 survey.

It plans two receiver-product families:

1. **Nodal stack products** indexed by notebook 96.
2. **Original Geode/land-streamer SEG-2 `.dat` gathers from `GEODE_DATA/LBS_*`**, discovered from the raw T1
   acquisition directory using the `survey` and `geode_file_no` fields.

The notebook does not alter or rewrite any waveform file.

## Main outputs

- `98_virtual_T1_shot_catalog.csv`
- `98_virtual_T1_product_plan.csv`
- `98_virtual_T1_pair_equations.csv`
- `98_virtual_T1_shift_solution.csv`
- `98_virtual_T1_geode_file_catalog.csv`
- `98_virtual_T1_plan_summary.csv`

The shift solution uses all accepted pairwise lag equations inside a source
cluster and constrains the canonical stack to zero. For a pair `left` versus
`right`, notebook 97's positive lag is interpreted as:

```text
shift_right - shift_left = measured_lag
```

where a positive shift moves the product later on the virtual relative-time axis.

## 1. Configuration

In [ ]:
from pathlib import Path
import re
from collections import defaultdict, deque

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')

MANIFEST_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_manifest'
REVIEW_ROOT = PROJECT_ROOT / '97_nodal_source_cluster_review'
OUT_ROOT = PROJECT_ROOT / '98_virtual_T1_integration_plan'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

STACK_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_manifest.csv'
FILE_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_file_manifest.csv'
CLUSTER_PATH = REVIEW_ROOT / '97_source_clusters.csv'
MEMBERSHIP_PATH = REVIEW_ROOT / '97_source_cluster_membership.csv'
PAIR_REVIEW_PATH = REVIEW_ROOT / '97_candidate_pair_review.csv'

TARGET_LINE = 'T1'
COMPONENT = 'Z'

# Original Geode/land-streamer SEG-2 files are stored by acquisition
# date under GEODE_DATA/LBS_MMDDYY. T1 was acquired on 05/18/26, so that
# folder is preferred, but all LBS_* folders are searched as a fallback.
GEODE_DATA_ROOT = PROJECT_ROOT / 'GEODE_DATA'
PREFERRED_GEODE_DATE_DIRS = ['LBS_051826']
SEARCH_ALL_LBS_DATE_DIRS = True

preferred_roots = [
    GEODE_DATA_ROOT / dirname
    for dirname in PREFERRED_GEODE_DATE_DIRS
]
other_roots = (
    sorted(GEODE_DATA_ROOT.glob('LBS_*'))
    if SEARCH_ALL_LBS_DATE_DIRS and GEODE_DATA_ROOT.exists()
    else []
)
RAW_GEODE_ROOTS = []
for root in [*preferred_roots, *other_roots]:
    if root not in RAW_GEODE_ROOTS:
        RAW_GEODE_ROOTS.append(root)

# These Geometrics SEG-2 records use .dat. Additional extensions remain
# configurable, but .dat is the expected primary format.
RAW_GEODE_EXTENSIONS = {'.dat'}

# Only a uniquely best match is accepted. Exact filename-stem matches are
# strongly preferred over a record number appearing elsewhere in the name.
REQUIRE_UNIQUE_BEST_GEODE_MATCH = True

# Pair acceptance policy.
ACCEPT_AUTOMATIC_STATUSES = {'waveform_match_supported'}
INCLUDE_INCONCLUSIVE_PAIRS = False
PAIR_WEIGHT_COLUMN = 'median_envelope_corrcoef'
MIN_PAIR_WEIGHT = 0.05

# Raw Geode receiver geometry fallback when headers do not contain x coordinates.
# These values match the established T1 Geode geometry used elsewhere.
GEODE_FIRST_RECEIVER_X_M = 87.0
GEODE_RECEIVER_DX_M = 1.0
GEODE_REVERSE_TRACE_ORDER = False

# Raw Geode products are included when a file can be found.
INCLUDE_RAW_GEODE = True
REQUIRE_ALL_RAW_GEODE_FILES = False

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Output:', OUT_ROOT)
print('Geode data root:', GEODE_DATA_ROOT)
print('Raw Geode roots (in preference order):')
for root in RAW_GEODE_ROOTS:
    print(' ', root, 'exists=', root.exists())

## 2. Load notebook 96 and 97 products

In [ ]:
required_paths = [
    STACK_MANIFEST_PATH,
    FILE_MANIFEST_PATH,
    CLUSTER_PATH,
    MEMBERSHIP_PATH,
    PAIR_REVIEW_PATH,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing prerequisite files:\n'
        + '\n'.join(f'  {path}' for path in missing)
    )

stacks = pd.read_csv(STACK_MANIFEST_PATH, low_memory=False)
files = pd.read_csv(FILE_MANIFEST_PATH, low_memory=False)
clusters = pd.read_csv(CLUSTER_PATH, low_memory=False)
membership = pd.read_csv(MEMBERSHIP_PATH, low_memory=False)
pairs = pd.read_csv(PAIR_REVIEW_PATH, low_memory=False)

for frame in [stacks, clusters, membership, pairs]:
    if 'line' in frame:
        frame['line'] = frame['line'].astype(str)

stacks['source_x_m'] = pd.to_numeric(stacks['source_x_m'], errors='coerce')
membership['source_x_m'] = pd.to_numeric(
    membership['source_x_m'], errors='coerce'
)
clusters['canonical_source_x_m'] = pd.to_numeric(
    clusters['canonical_source_x_m'], errors='coerce'
)

stacks_t1 = stacks.loc[stacks.line.eq(TARGET_LINE)].copy()
membership_t1 = membership.loc[membership.line.eq(TARGET_LINE)].copy()
clusters_t1 = clusters.loc[clusters.line.eq(TARGET_LINE)].copy()
pairs_t1 = pairs.loc[pairs.line.eq(TARGET_LINE)].copy()

print('T1 stack products:', len(stacks_t1))
print('T1 source clusters:', len(clusters_t1))
print('T1 pair reviews:', len(pairs_t1))

## 3. Build ordered virtual-shot catalog

In [ ]:
shot_catalog = (
    clusters_t1
    .sort_values(
        ['canonical_source_x_m', 'source_cluster_id'],
        kind='stable',
    )
    .reset_index(drop=True)
)

shot_catalog.insert(
    0,
    'virtual_shot_number',
    np.arange(1, len(shot_catalog) + 1, dtype=int),
)
shot_catalog.insert(
    1,
    'virtual_shot_id',
    [
        f'T1_VSHOT_{number:04d}_x{source_x:07.1f}m'
        for number, source_x in zip(
            shot_catalog.virtual_shot_number,
            shot_catalog.canonical_source_x_m,
        )
    ],
)

shot_catalog['source_x_m'] = shot_catalog['canonical_source_x_m']
shot_catalog['component'] = COMPONENT

print('Ordered virtual T1 shots:', len(shot_catalog))
display(
    shot_catalog[
        [
            'virtual_shot_number', 'virtual_shot_id',
            'source_cluster_id', 'source_x_m',
            'n_stack_products', 'canonical_stack_id',
        ]
    ].head(20)
)

## 4. Resolve notebook-96 nodal stack files

In [ ]:
mseed_files = files.loc[
    files.file_type.astype(str).str.lower().eq('mseed')
    & files.component.astype(str).str.upper().eq(COMPONENT)
    & files.file_exists.astype(str).str.lower().isin(
        ['true', '1', 'yes']
    )
].copy()

if mseed_files.stack_id.duplicated().any():
    duplicate_rows = mseed_files.loc[
        mseed_files.stack_id.duplicated(keep=False)
    ].sort_values('stack_id')
    display(duplicate_rows)
    raise RuntimeError(
        'More than one existing Z MiniSEED product was indexed for a stack.'
    )

stack_file_lookup = dict(
    zip(mseed_files.stack_id.astype(str), mseed_files.file_path.astype(str))
)

stack_details = stacks_t1.set_index(
    stacks_t1.stack_id.astype(str),
    drop=False,
)

nodal_product_rows = []

for member in membership_t1.itertuples(index=False):
    stack_id = str(member.stack_id)
    if stack_id not in stack_file_lookup:
        continue

    details = stack_details.loc[stack_id]
    if isinstance(details, pd.DataFrame):
        details = details.iloc[0]

    shot = shot_catalog.loc[
        shot_catalog.source_cluster_id.eq(member.source_cluster_id)
    ].iloc[0]

    nodal_product_rows.append({
        'virtual_shot_number': int(shot.virtual_shot_number),
        'virtual_shot_id': shot.virtual_shot_id,
        'source_cluster_id': member.source_cluster_id,
        'source_x_m': float(shot.source_x_m),
        'product_id': stack_id,
        'product_kind': 'nodal_stack',
        'receiver_family': 'nodal',
        'catalog_branch': member.catalog_branch,
        'survey': details.get('survey', np.nan),
        'priority': int(member.priority),
        'merge_stage': member.merge_stage,
        'component': COMPONENT,
        'waveform_path': stack_file_lookup[stack_id],
        'geode_file_no': pd.to_numeric(
            details.get('geode_file_no', np.nan), errors='coerce'
        ),
        'n_accepted_members': pd.to_numeric(
            details.get('n_accepted_members', np.nan), errors='coerce'
        ),
        'is_canonical_stack': bool(member.is_canonical_stack),
    })

nodal_products = pd.DataFrame(nodal_product_rows)
print('Planned nodal stack products:', len(nodal_products))

## 5. Discover original Geode/streamer files

In [ ]:
def extract_numeric_tokens(path):
    return {
        int(token)
        for token in re.findall(r'(?<!\d)(\d{3,6})(?!\d)', path.stem)
    }


raw_candidates = []
for root in RAW_GEODE_ROOTS:
    if not root.exists():
        continue
    for path in root.rglob('*'):
        if not path.is_file():
            continue
        if path.suffix.lower() not in RAW_GEODE_EXTENSIONS:
            continue
        raw_candidates.append({
            'raw_path': str(path),
            'filename': path.name,
            'numeric_tokens': extract_numeric_tokens(path),
            'size_bytes': path.stat().st_size,
        })

print('Raw Geode/streamer candidate files:', len(raw_candidates))

geode_requirements = (
    nodal_products.loc[
        nodal_products.geode_file_no.notna(),
        [
            'virtual_shot_number', 'virtual_shot_id',
            'source_cluster_id', 'source_x_m',
            'survey', 'geode_file_no',
        ],
    ]
    .drop_duplicates()
    .copy()
)
geode_requirements['geode_file_no'] = (
    geode_requirements.geode_file_no.astype(int)
)

geode_catalog_rows = []

for requirement in geode_requirements.itertuples(index=False):
    matches = [
        candidate
        for candidate in raw_candidates
        if int(requirement.geode_file_no) in candidate['numeric_tokens']
    ]

    matches = sorted(
        matches,
        key=lambda item: (
            len(Path(item['raw_path']).parts),
            item['filename'],
            item['raw_path'],
        ),
    )

    selected = matches[0]['raw_path'] if len(matches) == 1 else None
    status = (
        'matched_unique'
        if len(matches) == 1
        else 'missing'
        if len(matches) == 0
        else 'ambiguous'
    )

    geode_catalog_rows.append({
        'virtual_shot_number': requirement.virtual_shot_number,
        'virtual_shot_id': requirement.virtual_shot_id,
        'source_cluster_id': requirement.source_cluster_id,
        'source_x_m': requirement.source_x_m,
        'survey': requirement.survey,
        'geode_file_no': requirement.geode_file_no,
        'match_status': status,
        'n_candidate_files': len(matches),
        'selected_raw_path': selected,
        'candidate_paths': ' | '.join(
            item['raw_path'] for item in matches
        ),
    })

geode_file_catalog = pd.DataFrame(geode_catalog_rows)

if len(geode_file_catalog):
    display(
        geode_file_catalog.groupby(
            ['survey', 'match_status'],
            dropna=False,
        ).size().reset_index(name='n_files')
    )

unresolved_geode = geode_file_catalog.loc[
    ~geode_file_catalog.match_status.isin(['matched_unique', 'matched_first_best'])
].copy()

if len(unresolved_geode):
    print('Unresolved original Geode/streamer files:', len(unresolved_geode))
    display(unresolved_geode.head(50))
    if REQUIRE_ALL_RAW_GEODE_FILES:
        raise RuntimeError(
            'Some original Geode/streamer files were missing or ambiguous.'
        )

## 6. Add original Geode/streamer receiver products

In [ ]:
geode_product_rows = []

if INCLUDE_RAW_GEODE and len(geode_file_catalog):
    for row in geode_file_catalog.loc[
        geode_file_catalog.match_status.isin(['matched_unique', 'matched_first_best'])
    ].itertuples(index=False):
        geode_product_rows.append({
            'virtual_shot_number': int(row.virtual_shot_number),
            'virtual_shot_id': row.virtual_shot_id,
            'source_cluster_id': row.source_cluster_id,
            'source_x_m': float(row.source_x_m),
            'product_id': (
                f'RAW_GEODE_{row.survey}_F{int(row.geode_file_no):04d}'
            ),
            'product_kind': 'geode_raw',
            'receiver_family': 'geode',
            'catalog_branch': 'original_geode_receiver_gather',
            'survey': row.survey,
            'priority': 0,
            'merge_stage': 'original_receiver_data',
            'component': COMPONENT,
            'waveform_path': row.selected_raw_path,
            'acquisition_date_dir': row.selected_acquisition_date_dir,
            'raw_file_match_reason': row.selected_match_reason,
            'raw_file_match_score': row.best_match_score,
            'geode_file_no': int(row.geode_file_no),
            'n_accepted_members': 1,
            'is_canonical_stack': False,
        })

geode_products = pd.DataFrame(geode_product_rows)
print('Planned original Geode/streamer products:', len(geode_products))

## 7. Select pair equations and solve product shifts

In [ ]:
accepted_pairs = pairs_t1.loc[
    pairs_t1.automatic_status.isin(ACCEPT_AUTOMATIC_STATUSES)
].copy()

if INCLUDE_INCONCLUSIVE_PAIRS:
    accepted_pairs = pairs_t1.loc[
        pairs_t1.automatic_status.isin(
            ACCEPT_AUTOMATIC_STATUSES
            | {'waveform_match_inconclusive'}
        )
    ].copy()

accepted_pairs['measured_lag_s'] = pd.to_numeric(
    accepted_pairs['gather_lag_s'], errors='coerce'
)
accepted_pairs['equation_weight'] = pd.to_numeric(
    accepted_pairs.get(PAIR_WEIGHT_COLUMN, 1.0),
    errors='coerce',
).fillna(1.0).clip(lower=MIN_PAIR_WEIGHT)

pair_equations = accepted_pairs[
    [
        'comparison_id', 'source_cluster_id',
        'left_stack_id', 'right_stack_id',
        'measured_lag_s', 'equation_weight',
        'automatic_status',
    ]
].copy()
pair_equations['equation'] = (
    'shift(' + pair_equations.right_stack_id.astype(str)
    + ') - shift(' + pair_equations.left_stack_id.astype(str)
    + ') = ' + pair_equations.measured_lag_s.map(
        lambda value: f'{value:.9f}'
    )
)


def solve_cluster_shifts(member_frame, equation_frame):
    member_frame = member_frame.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    ).copy()

    ids = member_frame.stack_id.astype(str).tolist()
    canonical_candidates = member_frame.loc[
        member_frame.is_canonical_stack.astype(bool)
    ]
    canonical_id = (
        str(canonical_candidates.iloc[0].stack_id)
        if len(canonical_candidates)
        else ids[0]
    )

    if len(ids) == 1:
        return pd.DataFrame([{
            'stack_id': ids[0],
            'time_shift_to_canonical_s': 0.0,
            'shift_solution_status': 'single_product',
            'shift_residual_rms_s': 0.0,
            'n_shift_equations': 0,
        }])

    index = {stack_id: i for i, stack_id in enumerate(ids)}
    rows = []
    rhs = []
    weights = []

    for equation in equation_frame.itertuples(index=False):
        left = str(equation.left_stack_id)
        right = str(equation.right_stack_id)
        if left not in index or right not in index:
            continue
        vector = np.zeros(len(ids), dtype=float)
        vector[index[right]] = 1.0
        vector[index[left]] = -1.0
        rows.append(vector)
        rhs.append(float(equation.measured_lag_s))
        weights.append(float(equation.equation_weight))

    # Canonical constraint shift(canonical) = 0.
    constraint = np.zeros(len(ids), dtype=float)
    constraint[index[canonical_id]] = 1.0
    rows.append(constraint)
    rhs.append(0.0)
    weights.append(max(weights, default=1.0) * 1000.0)

    matrix = np.vstack(rows)
    rhs = np.asarray(rhs, dtype=float)
    weights = np.sqrt(np.asarray(weights, dtype=float))

    weighted_matrix = matrix * weights[:, None]
    weighted_rhs = rhs * weights
    solution, _, rank, _ = np.linalg.lstsq(
        weighted_matrix,
        weighted_rhs,
        rcond=None,
    )

    data_matrix = matrix[:-1]
    data_rhs = rhs[:-1]
    if len(data_rhs):
        residuals = data_matrix @ solution - data_rhs
        residual_rms = float(np.sqrt(np.mean(residuals ** 2)))
    else:
        residual_rms = np.nan

    status = (
        'solved_connected'
        if rank >= len(ids)
        else 'underdetermined'
    )

    return pd.DataFrame([
        {
            'stack_id': stack_id,
            'time_shift_to_canonical_s': float(solution[index[stack_id]]),
            'shift_solution_status': status,
            'shift_residual_rms_s': residual_rms,
            'n_shift_equations': len(data_rhs),
        }
        for stack_id in ids
    ])


shift_frames = []

for cluster_id, member_frame in membership_t1.groupby(
    'source_cluster_id', sort=False
):
    equations = pair_equations.loc[
        pair_equations.source_cluster_id.eq(cluster_id)
    ]
    solved = solve_cluster_shifts(member_frame, equations)
    solved['source_cluster_id'] = cluster_id
    shift_frames.append(solved)

shift_solution = pd.concat(
    shift_frames,
    ignore_index=True,
) if shift_frames else pd.DataFrame()

display(
    shift_solution.groupby(
        ['shift_solution_status'], dropna=False
    ).size().reset_index(name='n_products')
)

## 8. Assemble final product plan

In [ ]:
product_plan = pd.concat(
    [nodal_products, geode_products],
    ignore_index=True,
    sort=False,
)

product_plan = product_plan.merge(
    shift_solution[
        [
            'source_cluster_id', 'stack_id',
            'time_shift_to_canonical_s',
            'shift_solution_status',
            'shift_residual_rms_s',
            'n_shift_equations',
        ]
    ],
    left_on=['source_cluster_id', 'product_id'],
    right_on=['source_cluster_id', 'stack_id'],
    how='left',
)

# Original Geode receiver gathers use the timing of their corresponding
# geode-linked nodal stack. Join by cluster, survey, and file number.
geode_shift_lookup = nodal_products.merge(
    shift_solution,
    left_on=['source_cluster_id', 'product_id'],
    right_on=['source_cluster_id', 'stack_id'],
    how='left',
)[
    [
        'source_cluster_id', 'survey', 'geode_file_no',
        'time_shift_to_canonical_s',
        'shift_solution_status',
        'shift_residual_rms_s',
        'n_shift_equations',
    ]
].drop_duplicates()

for index, row in product_plan.loc[
    product_plan.product_kind.eq('geode_raw')
].iterrows():
    matches = geode_shift_lookup.loc[
        geode_shift_lookup.source_cluster_id.eq(row.source_cluster_id)
        & geode_shift_lookup.survey.astype(str).eq(str(row.survey))
        & geode_shift_lookup.geode_file_no.eq(row.geode_file_no)
    ]
    if len(matches):
        match = matches.iloc[0]
        for column in [
            'time_shift_to_canonical_s',
            'shift_solution_status',
            'shift_residual_rms_s',
            'n_shift_equations',
        ]:
            product_plan.loc[index, column] = match[column]

product_plan['time_shift_to_canonical_s'] = (
    pd.to_numeric(
        product_plan.time_shift_to_canonical_s,
        errors='coerce',
    )
)

# Single/isolated products that are canonical require no shift.
product_plan.loc[
    product_plan.time_shift_to_canonical_s.isna()
    & product_plan.is_canonical_stack.fillna(False),
    'time_shift_to_canonical_s',
] = 0.0

product_plan['include_in_virtual_shot'] = (
    product_plan.waveform_path.notna()
    & product_plan.time_shift_to_canonical_s.notna()
)

product_plan['receiver_first_x_m_fallback'] = np.where(
    product_plan.product_kind.eq('geode_raw'),
    GEODE_FIRST_RECEIVER_X_M,
    np.nan,
)
product_plan['receiver_dx_m_fallback'] = np.where(
    product_plan.product_kind.eq('geode_raw'),
    GEODE_RECEIVER_DX_M,
    np.nan,
)
product_plan['reverse_trace_order_fallback'] = np.where(
    product_plan.product_kind.eq('geode_raw'),
    GEODE_REVERSE_TRACE_ORDER,
    False,
)

product_plan = product_plan.sort_values(
    [
        'virtual_shot_number', 'receiver_family',
        'priority', 'product_id',
    ],
    kind='stable',
).reset_index(drop=True)

print('Planned waveform products:', len(product_plan))
print(
    'Included products:',
    int(product_plan.include_in_virtual_shot.sum()),
)
display(
    product_plan.groupby(
        [
            'product_kind', 'survey',
            'include_in_virtual_shot',
            'shift_solution_status',
        ],
        dropna=False,
    ).size().reset_index(name='n_products')
)

## 9. Integrity checks and export

In [ ]:
issues = []

if shot_catalog.virtual_shot_number.duplicated().any():
    issues.append('Duplicate virtual_shot_number values.')

if shot_catalog.source_x_m.isna().any():
    issues.append('At least one virtual shot has no source coordinate.')

included = product_plan.loc[
    product_plan.include_in_virtual_shot
]
missing_paths = included.loc[
    ~included.waveform_path.map(lambda value: Path(str(value)).exists())
]
if len(missing_paths):
    issues.append(
        f'{len(missing_paths)} included waveform paths do not exist.'
    )

underdetermined = product_plan.loc[
    product_plan.product_kind.eq('nodal_stack')
    & product_plan.shift_solution_status.eq('underdetermined')
]
if len(underdetermined):
    issues.append(
        f'{len(underdetermined)} nodal products have underdetermined shifts.'
    )

print('Plan issues:', len(issues))
for issue in issues:
    print(' -', issue)

OUTPUTS = {
    'shots': OUT_ROOT / '98_virtual_T1_shot_catalog.csv',
    'products': OUT_ROOT / '98_virtual_T1_product_plan.csv',
    'equations': OUT_ROOT / '98_virtual_T1_pair_equations.csv',
    'shifts': OUT_ROOT / '98_virtual_T1_shift_solution.csv',
    'geode_files': OUT_ROOT / '98_virtual_T1_geode_file_catalog.csv',
    'summary': OUT_ROOT / '98_virtual_T1_plan_summary.csv',
}

shot_catalog.to_csv(OUTPUTS['shots'], index=False)
product_plan.to_csv(OUTPUTS['products'], index=False)
pair_equations.to_csv(OUTPUTS['equations'], index=False)
shift_solution.to_csv(OUTPUTS['shifts'], index=False)
geode_file_catalog.to_csv(OUTPUTS['geode_files'], index=False)

summary = pd.DataFrame([
    ('virtual_T1_shots', len(shot_catalog)),
    ('planned_products', len(product_plan)),
    ('included_products', int(product_plan.include_in_virtual_shot.sum())),
    ('nodal_stack_products', int(product_plan.product_kind.eq('nodal_stack').sum())),
    ('raw_geode_products', int(product_plan.product_kind.eq('geode_raw').sum())),
    ('included_raw_geode_products', int((product_plan.product_kind.eq('geode_raw') & product_plan.include_in_virtual_shot).sum())),
    ('raw_geode_candidate_dat_files', len(raw_candidates)),
    ('unresolved_raw_geode_files', len(unresolved_geode)),
    ('accepted_pair_equations', len(pair_equations)),
    ('plan_issue_count', len(issues)),
], columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for name, path in OUTPUTS.items():
    print(f'  {name:12s} {path}')

## 10. Next step

Notebook 99 reads the product plan, applies each product's constant shift on a
common relative-time grid, forms the receiver union, combines duplicate nodal
receivers using stack-member weights, retains Geode and nodal receiver families
separately, and writes ordered per-shot MiniSEED files plus provenance catalogs.